# HURP — Conflict × Agriculture Panel · Analysis Notebook

Global **admin-2 district × year** panel (1989–2025), 1,825,173 rows × **61 columns**, linking political violence to agricultural output and socioeconomic drivers.

- Full column definitions: `docs/CODEBOOK.md` · Sources & licenses: `docs/DATA_SOURCES.md`
- **Private data**: the full export contains ACLED-derived columns (EULA forbids public sharing). Keep this repo private.
- `NaN` means *not observed*, **never** zero — see the careful-use notes at the bottom.

This notebook runs as-is in Google Colab or locally.

## 1 · Load the data

Pick whichever applies. In Colab the easiest is to **upload** `data/published/hurp_panel_v0.2_full.csv.gz`, or clone the private repo with a GitHub token.

In [ ]:
import os, pandas as pd, numpy as np
pd.set_option("display.max_columns", 80); pd.set_option("display.width", 200)

# Option A — local / repo checkout (looks in common locations):
CANDS = ["hurp_panel_v0.2_full.csv.gz",
         "data/published/hurp_panel_v0.2_full.csv.gz",
         "/content/hurp_panel_v0.2_full.csv.gz"]
path = next((p for p in CANDS if os.path.exists(p)), None)

# Option B — Colab upload (uncomment):
# from google.colab import files; up = files.upload(); path = next(iter(up))

# Option C — clone the PRIVATE repo (uncomment, paste a GitHub token):
# TOKEN = "ghp_xxx"
# !git clone https://{TOKEN}@github.com/nlethetech/HURP-project-.git
# path = "HURP-project-/data/published/hurp_panel_v0.2_full.csv.gz"

assert path, "No data file found — use Option B or C above."
df = pd.read_csv(path)
print(path, "->", df.shape)

## 2 · First look

In [ ]:
print("rows:", len(df), "| cols:", df.shape[1])
print("districts:", df.district_id.nunique(), "| countries:", df.iso3.nunique(),
      "| years:", df.year.min(), "-", df.year.max())
df.head(4)

## 3 · Column families & coverage

The 61 columns fall into families. `NaN` rates are meaningful (e.g. ACLED is only non-null inside each country's coverage window).

In [ ]:
families = {
 "keys": ["district_id","iso3","district_name","admin_level","year"],
 "UCDP conflict (fatal, zero-filled)": [c for c in df if c.startswith(("n_events","deaths_"))],
 "Coups (Powell & Thyne)": [c for c in df if c.startswith("coups_")],
 "ACLED political violence/unrest": [c for c in df if c.startswith("acled_")],
 "World Bank socioeconomic/ag": [c for c in df if c.startswith("wb_")],
 "Weather / yields / crop / price": ["precip_mm","yield_maize","yield_rice","yield_wheat",
        "yield_soybean","cropland_ha","price_shock_coverage","price_shock"],
 "Income": ["income_group","income_group_carried"],
}
for fam, cols in families.items():
    nn = np.mean([df[c].notna().mean() for c in cols if c in df]) * 100
    print(f"{fam:38s} {len(cols):2d} cols   mean non-null {nn:5.1f}%")

## 4 · The Maiduguri story (Boko Haram epicentre)

Borno State, Nigeria. The insurgency's 2009 outbreak and 2014–15 peak emerge straight from the panel — UCDP fatal events, ACLED's wider unrest, rising unemployment, and the country's income reclassification.

In [ ]:
MAID = "59680162B20780696540875"
m = df[df.district_id == MAID]
show = ["year","n_events_total","deaths_best_total","acled_events_total",
        "acled_events_protests","wb_unemployment","wb_gdp_pc","precip_mm",
        "yield_maize","income_group"]
m[m.year.isin([1995,2005,2009,2014,2015,2020,2024])][show].to_string(index=False)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13,4))
ax[0].bar(m.year, m.deaths_best_total, color="firebrick", label="UCDP deaths")
ax[0].set_title("Maiduguri — UCDP fatalities/yr"); ax[0].legend()
ax2 = ax[1]
ax2.plot(m.year, m.acled_events_total, "o-", label="ACLED events")
ax2.plot(m.year, m.n_events_total, "s-", label="UCDP fatal events")
ax2.set_title("Fatal (UCDP) vs all unrest (ACLED)"); ax2.legend()
plt.tight_layout(); plt.show()

## 5 · Global conflict over time

Country-year is the natural aggregation. Counts/deaths **sum**; ACLED is summed only where covered (its NaNs are not-observed).

In [ ]:
g = df.groupby("year").agg(
        ucdp_deaths=("deaths_best_total","sum"),
        ucdp_events=("n_events_total","sum"),
        acled_events=("acled_events_total","sum")).reset_index()
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(g.year, g.ucdp_deaths, color="firebrick", label="UCDP deaths")
ax.set_ylabel("UCDP deaths"); ax.legend(loc="upper left")
axb = ax.twinx(); axb.plot(g.year, g.acled_events, color="steelblue", label="ACLED events")
axb.set_ylabel("ACLED events"); axb.legend(loc="upper right")
ax.set_title("Global conflict, 1989–2025"); plt.show()

## 6 · Aggregating to country-year

A reusable helper: **sum** counts/areas, **cropland-weighted mean** for yields/precip/shock, **first** for national series (WB, income, coups).

In [ ]:
def to_country_year(d):
    cnt = [c for c in d if c.startswith(("n_events","deaths_","coups_","acled_"))]
    g = d.groupby(["iso3","year"])
    out = g[cnt].sum(min_count=1)
    # cropland-weighted means
    w = d["cropland_ha"].fillna(0)
    for col in ["precip_mm","yield_maize","price_shock"]:
        num = (d[col]*w).groupby([d.iso3,d.year]).sum()
        den = w.where(d[col].notna()).groupby([d.iso3,d.year]).sum()
        out[col] = num/den.replace(0,np.nan)
    nat = [c for c in d if c.startswith("wb_")] + ["income_group"]
    out = out.join(g[nat].first())
    return out.reset_index()

cy = to_country_year(df)
print(cy.shape); cy[cy.iso3=="NGA"].tail(3)[["iso3","year","deaths_best_total","wb_unemployment","wb_gdp_pc","yield_maize"]]

## 7 · Do socioeconomic drivers track conflict? (cross-section)

In [ ]:
sub = cy[(cy.year>=2000)].copy()
sub["confl"] = np.log1p(sub["deaths_best_total"])
corrs = sub[["confl","wb_unemployment","wb_unemployment_youth","wb_gdp_pc",
             "wb_gdp_growth","wb_pop_0_14","precip_mm"]].corr()["confl"].drop("confl")
print("Correlation of log(1+deaths) with drivers (country-year, 2000+):")
print(corrs.sort_values(ascending=False).round(3))

## 8 · A simple panel regression (illustrative)

Conflict on socioeconomic + climate drivers with **year fixed effects** (add country FE for a within estimate). This is a didactic example, not a finished causal model.

In [ ]:
import statsmodels.formula.api as smf
reg = cy.copy()
reg["log_deaths"] = np.log1p(reg["deaths_best_total"])
reg["gdp_k"] = reg["wb_gdp_pc"]/1000.0
reg = reg.dropna(subset=["log_deaths","wb_unemployment","gdp_k","wb_pop_0_14","precip_mm"])
m = smf.ols("log_deaths ~ wb_unemployment + gdp_k + wb_pop_0_14 + precip_mm + C(year)",
            data=reg).fit(cov_type="cluster", cov_kwds={"groups": reg["iso3"]})
print(m.summary().tables[1])
# For a within-country (FE) estimate, add + C(iso3) above, or use linearmodels PanelOLS:
# !pip install linearmodels -q

## 9 · Careful-use notes

- **`NaN` ≠ 0.** UCDP conflict and coups are zero-filled (globally complete → absence = true zero). Everything else is honest `NaN`:
  - `acled_*` is `NaN` **outside** each country's ACLED coverage window (staggered: Africa 1997 → USA 2020). `0` means covered-but-no-event. Don't sum NaN as 0 across the pre-coverage era.
  - `wb_*` is `NaN` where the World Bank has no observation (and the recent reporting lag leaves 2024–2025 sparse).
  - `yield_*` ends 2016 (GDHY); `precip_mm` is 50°S–50°N and ends 2024.
- **Coups & World Bank are national**, broadcast onto every district — don't sum them across districts; aggregate to country-year first.
- **Geometry is fixed** (geoBoundaries CGAZ v6.0.0); boundaries don't change across years.
- **License**: ACLED columns are for your own research only — keep the repo private. See `data/published/README.md`.